# Convolutional Neural Networks with PyTorch

This notebook demonstrates how to build and train CNNs using PyTorch as an alternative to TensorFlow/Keras.

## Key Differences:
- PyTorch uses NCHW format (batch, channels, height, width)
- Keras uses NHWC format (batch, height, width, channels)
- Explicit training loops instead of `model.fit()`
- `nn.Module` subclassing for model definition

In [ ]:
# Import PyTorch libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Import standard libraries
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Import utilities
from cnn_utils import *

%matplotlib inline
np.random.seed(1)
torch.manual_seed(1)

## 1. Load and Prepare Data

In [ ]:
# Load dataset
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_happy_dataset()

# Normalize
X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

# Reshape labels
Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print("X_train shape:", X_train.shape)  # (m, 64, 64, 3) - NHWC format
print("Y_train shape:", Y_train.shape)  # (m, 1)

## 2. Define CNN Model (PyTorch Style)

In [ ]:
class HappyModel(nn.Module):
    """
    CNN for binary classification
    Architecture: ZEROPAD -> CONV -> BATCHNORM -> RELU -> MAXPOOL -> FLATTEN -> FC
    """
    def __init__(self):
        super(HappyModel, self).__init__()
        
        # Define layers
        self.pad = nn.ZeroPad2d(3)                      # Padding: 3
        self.conv0 = nn.Conv2d(3, 32, 7, stride=1)      # Conv: 32 filters, 7x7, stride=1
        self.bn0 = nn.BatchNorm2d(32)                   # Batch Normalization
        self.relu = nn.ReLU()                           # ReLU activation
        self.pool0 = nn.MaxPool2d(2, stride=2)          # MaxPool: 2x2
        self.flatten = nn.Flatten()                     # Flatten
        self.fc = nn.Linear(32 * 32 * 32, 1)            # Fully connected: 32768 -> 1
        
    def forward(self, x):
        # Input x: (batch, 64, 64, 3) in NHWC format
        # Convert to NCHW format for PyTorch
        x = x.permute(0, 3, 1, 2)  # -> (batch, 3, 64, 64)
        
        # Forward pass
        x = self.pad(x)            # -> (batch, 3, 70, 70)
        x = self.conv0(x)          # -> (batch, 32, 64, 64)
        x = self.bn0(x)
        x = self.relu(x)
        x = self.pool0(x)          # -> (batch, 32, 32, 32)
        x = self.flatten(x)        # -> (batch, 32768)
        x = self.fc(x)             # -> (batch, 1)
        x = torch.sigmoid(x)       # Sigmoid for binary classification
        
        return x

# Create model
model = HappyModel()
print(model)

## 3. Prepare Data Loaders

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
Y_train_tensor = torch.FloatTensor(Y_train)
X_test_tensor = torch.FloatTensor(X_test)
Y_test_tensor = torch.FloatTensor(Y_test)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

## 4. Define Loss Function and Optimizer

In [ ]:
# Binary Cross Entropy Loss (equivalent to Keras 'binary_crossentropy')
criterion = nn.BCELoss()

# Adam optimizer (equivalent to Keras optimizer='adam')
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Optional: Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

## 5. Training Loop (Explicit - No model.fit())

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=10, device='cpu'):
    """
    Train a PyTorch model
    
    This replaces Keras' model.fit()
    """
    model = model.to(device)
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(X_batch)
            
            # Calculate loss
            loss = criterion(outputs, Y_batch)
            
            # Backward pass
            loss.backward()
            
            # Update weights
            optimizer.step()
            
            # Statistics
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_correct += (predicted == Y_batch).sum().item()
            train_total += Y_batch.size(0)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, Y_batch)
                
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == Y_batch).sum().item()
                val_total += Y_batch.size(0)
        
        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / len(test_loader)
        epoch_val_acc = val_correct / val_total
        
        # Store history
        history['loss'].append(epoch_train_loss)
        history['accuracy'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_accuracy'].append(epoch_val_acc)
        
        # Print progress
        print(f'Epoch [{epoch+1}/{epochs}], '
              f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, '
              f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')
    
    return history

In [ ]:
# Train the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

history = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=10, device=device)

## 6. Visualize Training History

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history['loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Model Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['accuracy'], label='Train Accuracy')
ax2.plot(history['val_accuracy'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Model Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Model Evaluation (Replaces model.evaluate())

In [ ]:
def evaluate_model(model, data_loader, criterion, device='cpu'):
    """
    Evaluate model performance
    
    Replaces Keras' model.evaluate()
    """
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, Y_batch in data_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, Y_batch)
            
            test_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == Y_batch).sum().item()
            total += Y_batch.size(0)
    
    avg_loss = test_loss / len(data_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy

# Evaluate on test set
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion, device)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

## 8. Save and Load Model

In [ ]:
# Save model (PyTorch way)
torch.save(model.state_dict(), 'happy_model.pth')
print("Model saved to 'happy_model.pth'")

# Load model (PyTorch way)
loaded_model = HappyModel()
loaded_model.load_state_dict(torch.load('happy_model.pth'))
loaded_model.eval()
print("Model loaded from 'happy_model.pth'")

## Summary: Keras vs PyTorch

### Key Differences:

| Aspect | Keras | PyTorch |
|--------|-------|----------|
| **Model Definition** | Sequential/Functional API | nn.Module subclassing |
| **Tensor Format** | NHWC (batch, height, width, channels) | NCHW (batch, channels, height, width) |
| **Training** | `model.fit(X, Y, epochs=10)` | Explicit training loop |
| **Evaluation** | `model.evaluate(X_test, Y_test)` | Custom evaluation function |
| **Compilation** | `model.compile(optimizer, loss, metrics)` | Define separately |
| **Data Loading** | `tf.data.Dataset` | `DataLoader` + `TensorDataset` |
| **Saving** | `model.save('model.h5')` | `torch.save(model.state_dict(), 'model.pth')` |
| **Loading** | `load_model('model.h5')` | `model.load_state_dict(torch.load('model.pth'))` |

### Advantages of PyTorch:
- More explicit and easier to debug
- Better for research and custom architectures
- Dynamic computation graphs
- Pythonic and intuitive API
- Better GPU utilization in some cases

### Advantages of Keras:
- Simpler and more concise
- Better for rapid prototyping
- Less boilerplate code
- Automatic batching and training loop